# 26 – Huấn luyện Random Forest bốn lớp (M2-R6a, tái lập)

## 1. Giới thiệu bài toán phân loại bốn trạng thái

Mỗi 10 s, firmware tính một vector **23 đặc trưng** góc (`svmg-angle-23-v1`) từ cửa sổ 60/180 s. Mô hình gán một trong bốn trạng thái
chuyển động, trùng `svmg_class_t`: `0 NORMAL`, `1 HIGH_ROLL` (biên độ lắc lớn), `2 PERSISTENT_OFFSET` (nghiêng lệch kéo dài),
`3 AMPLITUDE_GROWTH` (biên độ tăng dần). Dữ liệu là `SIMULATION`, nhãn là proxy `SIMULATION_SCENARIO_V2_CANDIDATE`: **không** phải
dự báo lật tàu hay đánh giá GM/GZ.

Notebook này **tái lập** quy trình M2-R6a bằng đúng mã `ml/train/rf_four_class_m2r6.py`: fit trên Train, chọn trên Validation, không
nạp Test-G1, không tạo thí nghiệm mới. Dữ liệu gitignored; nếu thiếu, chạy từ thư mục gốc repo:

```bash
.venv/bin/python -m ml.train.four_class_experimental_dataset   # dataset v0.2
.venv/bin/python -m ml.train.four_class_v03_dataset             # dataset v0.3
.venv/bin/python -m ml.train.rf_four_class_m2r6                 # mô hình M2-R6a (cần cho mục 6 và 13)
.venv/bin/python -m nbconvert --to notebook --execute ml/notebooks/26_four_class_rf_training.ipynb --output-dir ml/data/models/rf_four_class_m2r6/notebook_runs
```

In [ ]:
import json, sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / "AGENTS.md").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from ml.export import rf4_freeze_m2r6b as F
from ml.train import rf_four_class_m2r6 as R
from ml.train.g1_simulation_dataset import _sha

FIG = R.OUT / "notebook_runs" / "figures"
FIG.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 180)
for tag, (path, sha) in R.DATASETS.items():
    if not path.exists():
        raise FileNotFoundError(f"Thiếu dataset {tag} (gitignored): chạy các lệnh ở mục 1 từ thư mục gốc repo.")
    print(f"{tag}: {path.relative_to(ROOT)} | SHA-256 {sha[:16]}… | khớp bản đóng băng: {_sha(path) == sha}")
    assert _sha(path) == sha

## 2. Nguồn dữ liệu và cách sinh dataset

Bộ mô phỏng `simulation/generator` sinh roll 1-DOF theo kịch bản S0 (NORMAL), S1 (HIGH_ROLL), S2 (PERSISTENT_OFFSET), S3
(AMPLITUDE_GROWTH) và S0b (nghiêng tạm thời, tập challenge). IMU 20 Hz đi qua đúng đường Core C (lọc, hạ 10 Hz, cửa sổ 60/180 s) để
ra 23 đặc trưng. Nhãn gán theo **điểm cuối** cửa sổ với ngưỡng labeling-v2 candidate đóng băng; cửa sổ mơ hồ, khởi động, chuyển tiếp
hoặc đặc trưng không hợp lệ là `ABSTAIN` và không dùng để huấn luyện. Seed session băm SHA-256; manifest ghi SHA của dataset và nguồn.

In [ ]:
man = {t: json.loads((p.parent / "manifest.json").read_text()) for t, (p, _) in R.DATASETS.items()}
display(pd.DataFrame({t: {"dataset_version": m["dataset_version"], "sim_extension_version": m.get("sim_extension_version", "– (chỉ roll)"),
                         "imu_config": m.get("imu_config_name", "–"), "session": m["session_count"], "cửa sổ": m["window_count"],
                         "cửa sổ có nhãn": m["eligible_count"], "label_source": m["label_source"],
                         "feature_schema": m["feature_schema_version"], "preprocessing": m["preprocessing_version"]} for t, m in man.items()}))
print("Cấu hình nhiễu/pitch của v0.3:", man["v0.3"]["imu_config"])

## 3. So sánh dataset v0.2 và v0.3

v0.3 dùng **đúng** session, seed và nhãn của v0.2; chỉ thay mẫu IMU bằng bản có pitch và nhiễu/bias kiểu MPU6050
(`svmg-sim-pitch-imu-0.1.0`, `PITCH_MPU6050_RESIDUAL`). Vì vậy nhãn trùng 100 %, còn đặc trưng pitch không còn bằng 0 và đặc trưng
roll dịch nhẹ do bias/nhiễu.

In [ ]:
data = {t: R.load(p) for t, (p, _) in R.DATASETS.items()}   # chỉ cửa sổ có nhãn của Train/Validation; Test-G1 bị loại ngay khi đọc
a, b = data["v0.2"], data["v0.3"]
m = a.merge(b, on=["session_id", "window_end_seconds"], suffixes=("_v02", "_v03"))
print(f"Cửa sổ có nhãn Train+Validation: v0.2 {len(a)}, v0.3 {len(b)}, khớp khóa {len(m)}, trùng nhãn {np.mean(m.label_v02 == m.label_v03):.4f}")
q = lambda s: s.quantile([0.05, 0.5, 0.95]).round(4).tolist()
display(pd.DataFrame({"đặc trưng": ["pitch_std_60", "roll_mean_60"], "v0.2 (p5, p50, p95)": [q(a.pitch_std_60), q(a.roll_mean_60)],
                      "v0.3 (p5, p50, p95)": [q(b.pitch_std_60), q(b.roll_mean_60)]}))
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
for t, d in data.items():
    ax[0].hist(d.pitch_std_60, bins=40, alpha=0.6, label=t)
ax[0].set(title="pitch_std_60 trên cửa sổ có nhãn", xlabel="độ", ylabel="số cửa sổ"); ax[0].legend()
ax[1].hist(m.roll_mean_60_v03 - m.roll_mean_60_v02, bins=40, color="tab:purple")
ax[1].set(title="Chênh roll_mean_60 (v0.3 − v0.2)", xlabel="độ", ylabel="số cửa sổ")
fig.tight_layout(); fig.savefig(FIG / "26_v02_v03.png", dpi=110); plt.show()

## 4. Phân chia Train / Validation / Test-G1 và 5. Phân bố bốn lớp

Chia theo **session** (mọi cửa sổ của một session nằm trong một split). Test-G1 là đúng các session Test-G1 của pilot, giữ kín cho
đánh giá một lần (notebook 27); ở đây chỉ đọc **số đếm** từ manifest, không nạp đặc trưng hay nhãn Test-G1. S0b là tập challenge,
không dùng để huấn luyện.

In [ ]:
cls = pd.DataFrame([dict(zip(("tập", "split", "lớp"), k.split("|")), **{"cửa sổ": v["windows"], "session": v["sessions"],
                     "session hiệu dụng": v["effective_sessions"]}) for k, v in man["v0.3"]["classes"].items()])
display(cls.pivot_table(index="lớp", columns=["tập", "split"], values=["cửa sổ", "session"], fill_value=0).reindex(R.CLASSES))
print("Số session theo (split, tập):", dict(sorted(Counter((s["split"], s["subset"]) for s in man["v0.3"]["sessions"]).items())))
core = cls[cls["tập"] == "CORE"].pivot(index="lớp", columns="split", values="cửa sổ").reindex(R.CLASSES)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
core.plot.bar(ax=ax[0], rot=15, title="Cửa sổ có nhãn theo lớp và split (CORE)"); ax[0].set(xlabel="", ylabel="cửa sổ")
tv = core[["Train", "Validation"]]
(tv / tv.sum()).plot.bar(ax=ax[1], rot=15, title="Tỉ lệ lớp: Train và Validation"); ax[1].set(xlabel="", ylabel="tỉ lệ")
fig.tight_layout(); fig.savefig(FIG / "26_class_distribution.png", dpi=110); plt.show()

## 6. Giải thích vector 23 đặc trưng

- **Roll 60 s (biên độ):** RMS, độ lệch chuẩn, trung bình, đỉnh-đỉnh, p95/max trị tuyệt đối, IQR, RMS sai phân.
- **Roll 180 s (độ lệch và xu hướng):** trung bình/trung vị, tỉ lệ dấu trội, độ lệch giữa các đoạn, RMS đầu/cuối, tỉ số và độ dốc RMS,
  R² xu hướng, độ dốc p95.
- **Pitch 60 s:** RMS, độ lệch chuẩn, đỉnh-đỉnh (khác 0 từ v0.3).

Thứ tự là contract `contracts/features23.json` và phải trùng `SVMG_FEATURE_FIELDS` của Core C; lớp phải trùng `svmg_class_t`.

In [ ]:
feat = pd.DataFrame(json.loads((ROOT / "contracts/features23.json").read_text())["features"])[["index", "name", "unit"]]
feat["nhóm"] = np.select([feat.name.str.startswith("pitch"), feat.name.str.endswith("_180")], ["pitch 60 s", "roll 180 s"], "roll 60 s")
display(feat.set_index("index").T)
frozen, meta = F.load_frozen()          # dừng nếu SHA model.joblib khác bản đóng băng
print("Kiểm tra contract C (lớp, thứ tự đặc trưng, chỉ số nút):", R.c_contract_check(frozen))
assert all(R.c_contract_check(frozen).values()) and list(feat.name) == R.FEATURES == meta["feature_names"]

## 7. Kiểm tra dữ liệu và phòng tránh leakage

Đầu vào chỉ gồm 23 cột đặc trưng (float32 như ESP32); `target_coverage_180`, session, nhãn, kịch bản, split và provenance không vào
mô hình; NaN không bị thay bằng 0 (cửa sổ không hợp lệ đã là `ABSTAIN`).

In [ ]:
checks = {}
for t, d in data.items():
    tr, va = (d[d.subset.eq("CORE") & d.split.eq(s)] for s in (R.TRAIN, R.VAL))
    test_seeds = {s["seed"] for s in man[t]["sessions"] if s["split"] == "Test-G1"}
    checks[t] = {"không có Test-G1 trong dữ liệu đã nạp": not d.split.eq("Test-G1").any(),
                 "Train ∩ Validation (session) rỗng": not set(tr.session_id) & set(va.session_id),
                 "seed Test-G1 không trùng Train/Validation": not test_seeds & {s["seed"] for s in man[t]["sessions"] if s["split"] != "Test-G1"},
                 "ma trận 23 cột, hữu hạn": R.matrix(d).shape[1] == 23,
                 "đổi cột độ phủ/session/nhãn không đổi ma trận": np.array_equal(R.matrix(d), R.matrix(d.assign(
                     target_coverage_180=0.0, session_id="x", label="NORMAL", scenario="S0"))),
                 "cửa sổ Train CORE / Validation CORE": f"{len(tr)} / {len(va)}"}
display(pd.DataFrame(checks))

## 8. Các cấu hình Random Forest đã thử và 9. Quy trình huấn luyện, lựa chọn

Lưới **khai báo trước** 6 cấu hình (15/25/40 cây × độ sâu 4/6/8, mỗi cỡ có/không `balanced_subsample`), `max_features=sqrt`,
`random_state=20261124`. Fit trên CORE Train, đo macro-F1 trên CORE Validation; chọn macro-F1 cao nhất (làm tròn 3 chữ số), hòa thì
lấy cấu hình gọn hơn đứng trước. Ô dưới gọi lại `R.train_dataset` (không ghi artifact) và so với số đã công bố ở M2-R6a.

In [ ]:
res = {t: R.train_dataset(t, p, sha, None) for t, (p, sha) in R.DATASETS.items()}
trials = pd.DataFrame([{"dataset": t, "#": x["index"], **x["params"], "macro-F1 Validation": x["macro_f1"],
                        "recall lớp thấp nhất": x["min_class_recall"], "tổng nút": x["total_nodes"]} for t, r in res.items() for x in r["trials"]])
display(trials)
PUBLISHED = {"v0.2": 0.9112, "v0.3": 0.9876}   # ml/reports/ml/m2_r6a_four_class_rf_report.md
for t, r in res.items():
    print(f"{t}: chọn #{r['selected']['index']} {r['selected']['params']} | macro-F1 {r['validation']['macro_f1']} (công bố {PUBLISHED[t]})")
    assert r["selected"]["index"] == 0 and abs(r["validation"]["macro_f1"] - PUBLISHED[t]) <= 1e-4
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
for t, g in trials.groupby("dataset"):
    ax[0].plot(g["#"], g["macro-F1 Validation"], "o-", label=t)
    ax[1].scatter(g["tổng nút"], g["macro-F1 Validation"], label=t)
ax[0].set(title="macro-F1 Validation theo cấu hình", xlabel="cấu hình #", ylabel="macro-F1"); ax[0].legend()
ax[1].set(title="macro-F1 Validation và kích thước mô hình", xlabel="tổng số nút", ylabel="macro-F1"); ax[1].legend()
fig.tight_layout(); fig.savefig(FIG / "26_hyperparameters.png", dpi=110); plt.show()

## 10. So sánh macro-F1 v0.2/v0.3, 11. Confusion matrix Validation, 12. Precision/Recall/F1 từng lớp

Cùng 469 cửa sổ CORE Validation (12 session). Chênh lệch v0.2/v0.3 chủ yếu đến từ **một** session NORMAL bị đoán nhầm thành
AMPLITUDE_GROWTH; KTC bootstrap theo session chồng lấn nên chưa đủ bằng chứng nói pitch/nhiễu làm mô hình tốt hơn.

In [ ]:
display(pd.DataFrame({t: {k: r["validation"][k] for k in ("accuracy", "macro_f1", "weighted_f1", "macro_f1_session_bootstrap_95")}
                      for t, r in res.items()}))
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, (t, r) in zip(axes, res.items()):
    cm = np.array(r["validation"]["confusion"])
    ax.imshow(cm, cmap="Blues")
    for i, j in np.ndindex(cm.shape):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set(title=f"Confusion Validation {t}", xticks=range(4), yticks=range(4), xlabel="dự đoán", ylabel="thật")
    ax.set_xticklabels(R.CLASSES, rotation=20); ax.set_yticklabels(R.CLASSES)
fig.tight_layout(); fig.savefig(FIG / "26_confusion_validation.png", dpi=110); plt.show()
prf = pd.DataFrame([{"dataset": t, "lớp": c, **v} for t, r in res.items() for c, v in r["validation"]["per_class"].items()])
display(prf.pivot(index="lớp", columns="dataset", values=["precision", "recall", "f1", "support"]).reindex(R.CLASSES))
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6), sharey=True)
for ax, k in zip(axes, ("precision", "recall", "f1")):
    prf.pivot(index="lớp", columns="dataset", values=k).reindex(R.CLASSES).plot.bar(ax=ax, rot=20, title=k, ylim=(0, 1.05))
    ax.set_xlabel("")
fig.tight_layout(); fig.savefig(FIG / "26_prf_validation.png", dpi=110); plt.show()

## 13. Feature importance và giới hạn diễn giải

Fit lại cấu hình đã chọn trên CORE Train v0.3 phải cho **đúng** mô hình đóng băng: cùng cấu trúc cây và cùng xác suất. Bytes
pickle của mô hình fit trong notebook khác file đóng băng nên không so SHA joblib ở đây; SHA `3d7ecf31…` được tái lập bởi
`python -m ml.train.rf_four_class_m2r6` (đã kiểm trên bản clone sạch, M2-R8). Importance MDI
chỉ mô tả cách cây chia trên dữ liệu mô phỏng: đặc trưng tương quan chia nhau importance, MDI thiên về đặc trưng liên tục, và nhãn được
sinh từ ngưỡng trên roll nên importance phản ánh **luật proxy**, không phải cơ chế vật lý.

In [ ]:
d = data["v0.3"]
tr, va = (d[d.subset.eq("CORE") & d.split.eq(s)] for s in (R.TRAIN, R.VAL))
model = R.make_model(res["v0.3"]["selected"]["params"]).fit(R.matrix(tr), tr.y)
same_trees = all(np.array_equal(getattr(x.tree_, k), getattr(y.tree_, k)) for x, y in zip(model.estimators_, frozen.estimators_)
                 for k in ("feature", "threshold", "children_left", "children_right", "value"))
same = np.array_equal(model.predict_proba(R.matrix(va)), frozen.predict_proba(R.matrix(va)))
print(f"Cây trùng mô hình đóng băng (feature, ngưỡng, nhánh, giá trị lá): {same_trees} | xác suất Validation trùng: {same}")
assert same_trees and same
imp = pd.Series(model.feature_importances_, index=R.FEATURES).sort_values()
ax = imp.plot.barh(figsize=(7, 6.5), title="MDI importance – RF v0.3 đã chọn (15 cây, độ sâu 4)")
ax.set_xlabel("importance (tổng = 1)")
plt.tight_layout(); plt.savefig(FIG / "26_feature_importance.png", dpi=110); plt.show()

## 14. Lý do chọn RF 15 cây, độ sâu 4

- Theo quy tắc khai báo trước, cấu hình #0 có macro-F1 Validation **cao nhất** ở cả v0.2 và v0.3 (hòa với #4 ở v0.2 → lấy cấu hình gọn).
- Gọn nhất trong lưới: 185 nút (100 lá), tối đa 60 phép so sánh mỗi lần suy luận; mảng C ≈ 3,3 KB, object ESP32 `-Os` 3 624 B; Wokwi
  (SIMULATION) đo tối đa 719 µs mỗi lần suy luận.
- `balanced_subsample` kém hơn ở mọi kích thước; mô hình lớn hơn không cải thiện Validation.
- Chọn v0.3 vì quy tắc khai báo trước ưu tiên dữ liệu gần miền cảm biến hơn (pitch + nhiễu).

## 15. Kết luận và các hạn chế

Quy trình M2-R6a tái lập đúng: cùng dataset SHA, cùng cấu hình được chọn, cùng macro-F1 Validation và cùng mô hình đóng băng.
Hạn chế: nhãn proxy sinh từ ngưỡng trên chính bộ mô phỏng (RF học lại luật); Validation chỉ 12 session (AMPLITUDE_GROWTH 10 cửa
sổ/3 session) và đã dùng để chọn nên lạc quan; Test-G1 (notebook 27) thấp hơn rõ (macro-F1 hiệu dụng 0,816, precision
AMPLITUDE_GROWTH 0,34); chưa có dữ liệu MPU6050 thật hay kiểm chứng trên tàu.